# FlyRank AI — ML Internship Capstone (Final)
### Lane 2 · Refresh / Content Opportunity Scoring

**Title:** Prioritizing Content Reviews with Search Performance Signals: A Group-Validated Study

**Research question:** Can search-performance information already available at a decision date rank pages for review before a later impressions decline?

**Scope of the final research paper:** exactly **one fixed Random Forest** and **one transparent rule-based benchmark**, compared on the same cohort. **No leaderboard of the earlier experimental algorithms** is reported or exported. Ranked editorial review reasons, client-group validation, honest limitations, and the author’s identity section are included.

**Privacy:** This notebook reads authorized gated data through Google Colab; it does not publish tokens, client identifiers, URLs, queries, raw tables, or page-level review queues. The generated ZIP includes only the HTML paper, aggregate charts and aggregate metrics. No GitHub connection is necessary.

**Important historical disclosure:** The original April–May group holdout was already examined during earlier development. The notebook reports it as a historical audit, not an untouched confirmatory test.

## 0. Environment and editable public profile

Run in **Google Colab**. Enable the read-only Hugging Face token in the Colab Secrets panel as `HF_TOKEN` and grant notebook access. The first configuration cell below controls your **name, position, affiliation, biography, optional professional links, title, problem framing, and personal note**. Edit these fields before building the public paper. You can also manually edit `docs/index.html` after generation; rerunning the notebook overwrites manual HTML edits.

This notebook does not require `work/warehouse_loader.py`, `work/capstone_pipeline.py`, a GitHub clone, or a local CSV.

In [ ]:
%pip -q install "duckdb>=1.4,<2" "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.3" "matplotlib>=3.7"

In [ ]:
# Author and research-page settings; public-facing. Edit only if your publication identity changes. The HF token is never stored here.
from pathlib import Path
import json, os, sys, re, html, textwrap, zipfile
import numpy as np
import pandas as pd

REPO_URL = 'https://github.com/pretom26/ml_internship_flyrankAI'
DATA_RELEASE = 'v20260703'
FEATURE_MONTH = '2026-04'
OUTCOME_MONTH = '2026-05'
MIN_IMPRESSIONS = 100
MIN_GSC_DAYS = 5
SEED = 42
RUN_LATER_WINDOW_CHECK = True  # May -> June on the SAME predeclared holdout-client groups; no refitting or tuning.

AUTHOR = {
    'full_name': 'Pretom Areefin Pranto',
    'position': 'Machine Learning Intern — FlyRank AI',
    'affiliation': '4th Year Undergraduate CS Student at BRACU',
    'bio': ('This project examines how observational search-performance data can guide editorial priorities. ' 'The analysis uses a transparent ranking benchmark, a fixed Random Forest, and client-group validation ' 'to identify pages for evidence-based human review.'),
    'email': 'pretomareefin@gmail.com',
    'linkedin': 'https://www.linkedin.com/in/pretom-areefin-pranto-a97651290/',
    'portfolio': 'https://github.com/pretom26/pretom26/blob/main/Resume.pdf',
}

# Public research prose. Numerical results are always generated from the authorized Colab analysis.
PAPER_EDIT = {
    'title': 'Prioritizing Content Reviews with Search Performance Signals: A Group-Validated Study',
    'subtitle': 'A transparent rule-based score and a fixed Random Forest for next-month decline review',
    'problem_paragraph': (
        'Editorial teams cannot review every page at once. This study examines whether search-performance '
        'signals available at a decision date can prioritize which pages warrant diagnostic review for '
        'a substantial subsequent decline in search impressions.'
    ),
    'contribution_paragraph': (
        'The contribution is a reproducible, leakage-aware review-ranking workflow: a prespecified label, '
        'a transparent benchmark, one fixed machine-learning approach, group-level evaluation, and a '
        'reason-coded editorial action playbook.'
    ),
    'author_note': (
        'This internship project focuses on evidence-based decision support and reproducible analysis. '
        'It does not evaluate the causal effect of publishing or refreshing content.'
    ),
}

# Colab outputs go to a separate folder. The notebook is self-contained: no work/*.py imports or repo clone.
OUT = Path('/content/flyrank_final_capstone_output') if 'google.colab' in sys.modules else Path.cwd() / 'flyrank_final_capstone_output'
for subfolder in ('work/reports', 'docs/assets', 'submission'):
    (OUT / subfolder).mkdir(parents=True, exist_ok=True)
print('Output folder:', OUT)
print('Author:', AUTHOR['full_name'], '| No GitHub connection is needed.')

## 1. Read the FlyRank warehouse privately

**Source:** gated `FlyRank/internship-warehouse` dataset, pseudonymized fixed release v20260703. We use only the monthly partitions of `fact_content_daily_performance` and aggregate with DuckDB before transferring data into an in-memory pandas DataFrame.

**Primary feature window:** April 1–30, 2026; **outcome window:** May 1–31, 2026. Target label = May impressions per calendar day **more than 30% below** April impressions per calendar day. Eligible pairs need at least **100 April impressions** and **five recorded GSC days in both months**. Absent May measurement is *not* considered zero. Identifiers are used only for joining and group validation, never as predictors or public outputs.

In [ ]:
import getpass
import datetime as dt
import duckdb

WAREHOUSE = 'hf://datasets/FlyRank/internship-warehouse'

def secret_token():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        pass
    token = token or os.environ.get('HF_TOKEN')
    if not token:
        token = getpass.getpass('Hugging Face read token (input hidden): ')
    if not token.startswith('hf_'):
        raise ValueError('Read-only HF_TOKEN is required. Store it in Colab Secrets with notebook access enabled.')
    return token

def connect_hf():
    con = duckdb.connect()
    con.execute('INSTALL httpfs')
    con.execute('LOAD httpfs')
    # Reuse the exact secret-registration pattern that worked in the previous capstone.
    con.execute('SET VARIABLE flyrank_private_token = ?', [secret_token()])
    con.execute("CREATE OR REPLACE SECRET flyrank_read (TYPE huggingface, TOKEN getvariable('flyrank_private_token'))")
    con.execute('RESET VARIABLE flyrank_private_token')
    return con

def bounds(month):
    y,m = map(int, month.split('-'))
    start = dt.date(y,m,1)
    next_month = dt.date(y+(m==12),m%12+1,1)
    return start,next_month

def month_frame(con, feature_month, outcome_month, min_impressions=100, min_gsc_days=5):
    a,b = bounds(feature_month)
    c,d = bounds(outcome_month)
    if b != c or d > dt.date(2026,7,1):
        raise ValueError('Expected consecutive whole months within the FlyRank June 2026 data cutoff.')
    if min_impressions < 100 or min_gsc_days < 1:
        raise ValueError('Exposure threshold must be >=100 and observed GSC days >=1.')
    midpoint = dt.date(a.year,a.month,16)
    early_n, late_n = (midpoint-a).days, (b-midpoint).days
    feature_n, target_n = (b-a).days, (d-c).days
    src = lambda month: f"read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={month}/*.parquet')"
    feature_rel, outcome_rel = src(feature_month), src(outcome_month)
    columns = set(con.sql(f'DESCRIBE SELECT * FROM {feature_rel}').df().column_name)
    required={'client_hash_id','content_hash_id','report_date','gsc_impressions','gsc_clicks'}
    if not required.issubset(columns):
        raise RuntimeError('Unexpected warehouse columns: missing ' + str(sorted(required-columns)))
    if 'gsc_sum_position' in columns:
        position = 'SUM(COALESCE(gsc_sum_position,0)) / NULLIF(SUM(COALESCE(gsc_impressions,0)),0)'
    elif 'gsc_avg_position' in columns:
        position = 'SUM(COALESCE(gsc_avg_position,0)*COALESCE(gsc_impressions,0))/NULLIF(SUM(COALESCE(gsc_impressions,0)),0)'
    else:
        position = 'CAST(NULL AS DOUBLE)'
    query = f'''
    WITH features AS (
       SELECT client_hash_id,content_hash_id,
         SUM(COALESCE(gsc_impressions,0))::DOUBLE AS feature_impressions,
         SUM(COALESCE(gsc_clicks,0))::DOUBLE AS feature_clicks,
         {position} AS feature_position,
         COUNT(gsc_impressions)::INTEGER AS feature_observed_days,
         SUM(CASE WHEN report_date < DATE '{midpoint}' THEN COALESCE(gsc_impressions,0) ELSE 0 END)::DOUBLE AS early_impressions,
         SUM(CASE WHEN report_date >= DATE '{midpoint}' THEN COALESCE(gsc_impressions,0) ELSE 0 END)::DOUBLE AS late_impressions
       FROM {feature_rel}
       WHERE report_date >= DATE '{a}' AND report_date < DATE '{b}'
       GROUP BY 1,2
    ), targets AS (
       SELECT client_hash_id,content_hash_id,
         SUM(COALESCE(gsc_impressions,0))::DOUBLE AS outcome_impressions,
         COUNT(gsc_impressions)::INTEGER AS outcome_observed_days
       FROM {outcome_rel}
       WHERE report_date >= DATE '{c}' AND report_date < DATE '{d}'
       GROUP BY 1,2
    )
    SELECT p.client_hash_id AS client_id,p.content_hash_id AS content_id,
      p.feature_impressions,p.feature_clicks,p.feature_position,p.feature_observed_days,
      p.early_impressions,p.late_impressions,t.outcome_impressions,t.outcome_observed_days,
      (p.feature_clicks/NULLIF(p.feature_impressions,0))::DOUBLE AS feature_ctr,
      GREATEST(-1.0,LEAST(3.0,
        ((p.late_impressions/{late_n}.0) -(p.early_impressions/{early_n}.0)) /
        GREATEST(p.early_impressions/{early_n}.0,1.0)))::DOUBLE AS feature_momentum,
      CAST((t.outcome_impressions/{target_n}.0) < 0.70*(p.feature_impressions/{feature_n}.0) AS INTEGER) AS label
    FROM features p INNER JOIN targets t
      ON p.client_hash_id=t.client_hash_id AND p.content_hash_id=t.content_hash_id
    WHERE p.feature_impressions >= {int(min_impressions)}
      AND p.feature_observed_days >= {int(min_gsc_days)}
      AND t.outcome_observed_days >= {int(min_gsc_days)}
    '''
    frame = con.sql(query).df()
    if frame.empty:
        raise ValueError('No eligible rows for this month pair; inspect time coverage and thresholds.')
    frame['position_tier'] = frame['feature_position'].map(
        lambda v: 'unknown' if pd.isna(v) else ('top_3' if v<=3 else ('page_1' if v<=10 else ('page_2' if v<=20 else 'deep')))
    )
    meta = {
      'release':'FlyRank internship warehouse v20260703',
      'source_tables':['fact_content_daily_performance (month-partitioned)'],
      'feature_start':str(a),'feature_end_exclusive':str(b),
      'outcome_start':str(c),'outcome_end_exclusive':str(d),
      'feature_month':feature_month,'outcome_month':outcome_month,
      'eligibility':f'>= {min_impressions} feature-month impressions and >= {min_gsc_days} GSC-observed days in each month',
      'label':'Next-month impressions per calendar day >30% below feature-month impressions per calendar day',
      'excluded':'No next-month record, insufficient GSC observations, low feature exposure, missing identifiers or labels, duplicate client-content pairs',
      'missing_target':'Never assume a missing outcome-month row represents zero impressions',
      'feature_days':feature_n, 'outcome_days':target_n,
    }
    return frame,meta

con = connect_hf()
print('Authenticated. Querying aggregate monthly performance; no raw exports will be written.')
frame, DATA_META = month_frame(con, FEATURE_MONTH, OUTCOME_MONTH, MIN_IMPRESSIONS, MIN_GSC_DAYS)
print(f'Eligible observations: {len(frame):,}; client groups: {frame.client_id.nunique():,}; positive-label prevalence: {frame.label.mean():.3f}')
print('No hashed identifiers, row-level data, or secret values have been displayed.')

## 2. Features, the fixed Random Forest, and the rule benchmark

**Model:** the final, predetermined Random Forest uses 140 trees, depth at most 10, minimum leaf 35, `max_features='sqrt'`, and seed 42. This specification was informed by prior development; the final notebook **does not tune hyperparameters using holdout labels**.

**Baseline:** 40% capped log-exposure + 35% negative within-April momentum + 25% CTR shortfall against April train-client position peers. This transparent score does not use labels.

**Train/test:** 20% of unique pseudonymized clients by GroupShuffleSplit (seed 42), not 20% of rows. A training-only grouped validation provides a stability check. Preprocessing is inside the training pipeline. The final held-out comparison uses **the same pages and labels** for both methods. The historical cohort has already been examined; we disclose this rather than claiming untouched evaluation.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

NUMERIC = [
    'feature_impressions','feature_clicks','feature_ctr','feature_position',
    'feature_observed_days','feature_momentum','log_impressions','log_clicks',
    'impressions_per_observed_day','downward_momentum',
    'decline_x_log_exposure','ctr_x_log_exposure',
]
CATEGORICAL = ['position_tier']
PROHIBITED = {
    'client_id','content_id','label','outcome_impressions','outcome_observed_days',
    'target_impressions','target_observed_days','early_impressions','late_impressions',
}
assert set(NUMERIC+CATEGORICAL).isdisjoint(PROHIBITED)

def prepare_features(df):
    x = pd.DataFrame(index=df.index)
    for col in ['feature_impressions','feature_clicks','feature_ctr','feature_position',
                'feature_observed_days','feature_momentum']:
        x[col] = pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
    x['position_tier'] = df['position_tier'].fillna('unknown').astype(str)
    impressions = x.feature_impressions.clip(lower=0)
    clicks = x.feature_clicks.clip(lower=0)
    days = x.feature_observed_days.clip(lower=1)
    x['log_impressions'] = np.log1p(impressions)
    x['log_clicks'] = np.log1p(clicks)
    x['impressions_per_observed_day'] = impressions/days
    x['downward_momentum'] = (-x.feature_momentum).clip(lower=0,upper=1)
    x['decline_x_log_exposure'] = x.downward_momentum*x.log_impressions
    x['ctr_x_log_exposure'] = x.feature_ctr.clip(lower=0,upper=1)*x.log_impressions
    assert list(x.columns)==NUMERIC[:6]+['position_tier']+NUMERIC[6:] or set(x.columns)==set(NUMERIC+CATEGORICAL)
    return x[NUMERIC+CATEGORICAL]

def fixed_forest():
    transformer=ColumnTransformer([
        ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',RobustScaler())]),NUMERIC),
        ('cat',OneHotEncoder(handle_unknown='ignore'),CATEGORICAL),
    ],remainder='drop',sparse_threshold=0)
    return Pipeline([
        ('preprocess',transformer),
        ('model',RandomForestClassifier(
            n_estimators=140,max_depth=10,min_samples_leaf=35,
            max_features='sqrt',n_jobs=-1,random_state=SEED)),
    ])

def clean_data(df):
    required=set(['client_id','content_id','label','position_tier']+NUMERIC[:6])
    missing=required-set(df.columns)
    if missing:
        raise ValueError('Missing modeling columns: '+str(sorted(missing)))
    out=df.dropna(subset=['client_id','content_id','label']).copy()
    duplicates=int(out.duplicated(['client_id','content_id']).sum())
    if duplicates:
        raise ValueError(f'Duplicate client-content grain ({duplicates} rows); audit SQL before modeling.')
    if not out.label.isin([0,1]).all():
        raise ValueError('Outcome must be strictly binary 0/1')
    return out.reset_index(drop=True)

def select_client_split(df):
    groups=df.client_id.astype(str).to_numpy()
    idx_train,idx_test=next(GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED).split(df,df.label,groups))
    if set(groups[idx_train])&set(groups[idx_test]):
        raise AssertionError('Train/holdout client overlap')
    for idx in (idx_train,idx_test):
        if np.unique(df.label.to_numpy()[idx]).size!=2:
            raise ValueError('Fixed split is single-class; do not pick a seed using model test metrics.')
    return idx_train,idx_test

def rule_score(training_df,pred_df):
    # Declared rule weights fixed before accessing evaluation labels.
    imp_train=pd.to_numeric(training_df.feature_impressions,errors='coerce').clip(lower=0).fillna(0)
    imp_test=pd.to_numeric(pred_df.feature_impressions,errors='coerce').clip(lower=0).fillna(0)
    cap=max(float(imp_train.quantile(0.99)),1.)
    exposure=np.log1p(imp_test.clip(upper=cap))/np.log1p(cap)
    momentum=(-pd.to_numeric(pred_df.feature_momentum,errors='coerce').fillna(0)).clip(0,1)
    ctr_train=pd.to_numeric(training_df.feature_ctr,errors='coerce').fillna(0).clip(lower=0)
    fallback=float(ctr_train.median())
    peers=training_df.assign(_ctr=ctr_train).groupby('position_tier')['_ctr'].median()
    reference=pred_df.position_tier.map(peers).fillna(fallback).clip(lower=0.00001)
    observed_ctr=pd.to_numeric(pred_df.feature_ctr,errors='coerce').fillna(0).clip(lower=0)
    deficit=(1-observed_ctr/reference).clip(0,1).where(imp_test>=100,0.)
    return (0.40*exposure+0.35*momentum+0.25*deficit).to_numpy(float)

def p_at_k(y,scores,k):
    y=np.asarray(y,dtype=int);scores=np.asarray(scores,dtype=float)
    if len(y)<k:
        return float('nan')
    idx=np.argsort(-scores,kind='mergesort')[:k]
    return float(y[idx].mean())

def evaluate(y,scores):
    y=np.asarray(y,dtype=int)
    if np.unique(y).size!=2:raise ValueError('Metric evaluation needs both outcome classes.')
    scores=np.asarray(scores,dtype=float)
    result={'prevalence':float(y.mean()),
            'average_precision':float(average_precision_score(y,scores)),
            'roc_auc':float(roc_auc_score(y,scores))}
    for k in (10,20,50,100):
        result[f'precision_at_{k}']=p_at_k(y,scores,k)
        result[f'lift_at_{k}']=float(result[f'precision_at_{k}']/result['prevalence']) if result['prevalence'] else None
    return result

analysis=clean_data(frame)
idx_train,idx_test=select_client_split(analysis)
train=analysis.iloc[idx_train].copy()
holdout=analysis.iloc[idx_test].copy()
assert set(train.client_id).isdisjoint(set(holdout.client_id))

# Training-only grouped checks (not a multi-model search).
cv_records=[]
for i,(it,iv) in enumerate(GroupShuffleSplit(n_splits=3,test_size=0.25,random_state=2026).split(train,train.label,train.client_id),start=1):
    t=train.iloc[it];v=train.iloc[iv]
    if len(v)<100 or t.label.nunique()<2 or v.label.nunique()<2:
        continue
    model_cv=fixed_forest()
    model_cv.fit(prepare_features(t),t.label.to_numpy())
    rf_scores=model_cv.predict_proba(prepare_features(v))[:,1]
    rule_scores=rule_score(t,v)
    cv_records.append({'fold':i,'rf':evaluate(v.label,rf_scores),'rule':evaluate(v.label,rule_scores),
                       'clients_in_validation':int(v.client_id.nunique())})
if len(cv_records)<2:
    raise ValueError('Insufficient valid client-group validation folds to characterize model stability.')

# Model specification fixed BEFORE this holdout comparison; do not change it based on these results.
model=fixed_forest()
model.fit(prepare_features(train),train.label.to_numpy())
heldout_model_score=model.predict_proba(prepare_features(holdout))[:,1]
heldout_rule_score=rule_score(train,holdout)
RF_METRICS=evaluate(holdout.label,heldout_model_score)
RULE_METRICS=evaluate(holdout.label,heldout_rule_score)
print('Final study: ONE fixed Random Forest versus ONE rule-based baseline, same holdout rows.')
print('Rows:',len(analysis),'| train:',len(train),'| holdout:',len(holdout))
print('Client groups:',analysis.client_id.nunique(),'| held-out clients:',holdout.client_id.nunique())
for key in ('precision_at_20','precision_at_50','precision_at_100','average_precision','roc_auc'):
    print(f'{key:22s} Random Forest={RF_METRICS[key]:.4f}  Rule baseline={RULE_METRICS[key]:.4f}')

## 3. Uncertainty and later-period robustness

Cluster-bootstrap intervals resample held-out **client groups** and show uncertainty in model-minus-benchmark differences; the number of groups is limited.

Optionally the already-fitted model is scored on **May features → June outcomes** for the original held-out client groups, without hyperparameter changes or refitting. This later-label check reuses calendar-month information and is not an independent prospective trial. It is omitted with a clearly labeled note when data coverage is insufficient.

In [ ]:
# Cluster-level uncertainty: resample entire held-out clients, not individual pages.
# These intervals are descriptive (only ~10 held-out clients), not significance tests.
from collections import Counter

def paired_client_bootstrap(y,rf_scores,rule_scores,groups,n_resamples=400):
    y=np.asarray(y,dtype=int)
    rf_scores=np.asarray(rf_scores,dtype=float)
    rule_scores=np.asarray(rule_scores,dtype=float)
    groups=np.asarray(groups,dtype=str)
    rng=np.random.default_rng(2026)
    unique=np.unique(groups)
    index={g:np.where(groups==g)[0] for g in unique}
    storage={'precision_at_50':[],'average_precision':[],'roc_auc':[]}
    for _ in range(n_resamples):
        sampled=rng.choice(unique,size=len(unique),replace=True)
        inds=np.concatenate([index[g] for g in sampled])
        yy=y[inds]
        if len(inds)<50 or np.unique(yy).size<2:continue
        aa=evaluate(yy,rf_scores[inds]);bb=evaluate(yy,rule_scores[inds])
        for metric in storage:storage[metric].append(aa[metric]-bb[metric])
    return {k:{'delta':float(RF_METRICS[k]-RULE_METRICS[k]),
               'low_95':float(np.percentile(v,2.5)),
               'high_95':float(np.percentile(v,97.5)),
               'replicates':len(v)} for k,v in storage.items() if len(v)>30}

UNCERTAINTY=paired_client_bootstrap(holdout.label,heldout_model_score,heldout_rule_score,holdout.client_id)

# A later-window check uses May features and June outcomes on only the original held-out clients.
# The Random Forest and baseline peer references remain locked from April->May TRAIN clients.
LATER_RESULT={'status':'not_run','note':'Optional later-window robustness was disabled.'}
if RUN_LATER_WINDOW_CHECK:
    try:
        next_frame, LATER_META=month_frame(con,OUTCOME_MONTH,'2026-06',MIN_IMPRESSIONS,MIN_GSC_DAYS)
        next_frame=clean_data(next_frame)
        next_frame=next_frame[next_frame.client_id.isin(set(holdout.client_id))].copy()
        if len(next_frame)>=100 and next_frame.label.nunique()==2:
            next_rf=model.predict_proba(prepare_features(next_frame))[:,1]
            next_rule=rule_score(train,next_frame)
            LATER_RESULT={
               'status':'evaluated',
               'feature_window':'2026-05', 'label_window':'2026-06',
               'n_rows':int(len(next_frame)), 'n_groups':int(next_frame.client_id.nunique()),
               'rf_metrics':evaluate(next_frame.label,next_rf),
               'rule_metrics':evaluate(next_frame.label,next_rule),
               'note':('Same fixed model and original held-out client groups; no refitting/tuning. '
                       'Training labels from May are contemporaneous with the secondary check\'s May features, '
                       'but client groups are disjoint. This is a later-outcome robustness check, '
                       'not a wholly independent prospective trial.'),
            }
        else:
            LATER_RESULT={'status':'insufficient','n_rows':int(len(next_frame)),
                          'note':'Later period has insufficient eligible rows or both labels were not represented.'}
    except Exception as e:
        LATER_RESULT={'status':'unavailable','note':f'Later-window check unavailable: {type(e).__name__}; main evaluation is unaffected.'}

try:
    con.close()
except Exception:
    pass
print('Bootstrap intervals calculated for RF minus baseline (client-cluster resampling).')
for key,value in UNCERTAINTY.items():
    print(key,':',round(value['delta'],3), '95% descriptive interval',
          (round(value['low_95'],3),round(value['high_95'],3)))
print('Later-window status:',LATER_RESULT['status'])
if LATER_RESULT['status']=='evaluated':
    print('Later-window P@50 — RF:',round(LATER_RESULT['rf_metrics']['precision_at_50'],3),
          '| baseline:',round(LATER_RESULT['rule_metrics']['precision_at_50'],3))

## 4. Ranked action engine (private queue, public summaries)

We retain a **local-only** per-page ranked queue for authorized human review. The paper publishes only priority-band totals, reason-code counts, and an action playbook. The baseline determines which 50 pages are reviewed first; the Random Forest score provides supplementary risk evidence. A high score is **not evidence that changing a page will increase rankings or traffic**.

In [ ]:
# Private in-session queue only: no row-level identifiers or raw exports in the bundle.
def editorial_reason(row,training_data):
    impressions=float(row.feature_impressions)
    momentum=float(row.feature_momentum) if pd.notna(row.feature_momentum) else 0.
    if impressions < 200:
        return ('LOW_EVIDENCE','Monitor: obtain more search observations before making edits')
    if momentum <= -0.25:
        return ('VISIBLE_SLOWDOWN','Review demand shifts, indexing coverage, content currency and topical relevance')
    peer_ctr=training_data.assign(_ctr=pd.to_numeric(training_data.feature_ctr,errors='coerce')).groupby('position_tier')['_ctr'].median()
    peer=float(peer_ctr.get(row.position_tier,pd.to_numeric(training_data.feature_ctr,errors='coerce').median()))
    if pd.notna(peer) and peer>0 and float(row.feature_ctr)<0.70*peer:
        return ('BELOW_PEER_CTR','Check snippet/metadata relevance against search intent before rewriting')
    if pd.notna(row.feature_position) and 7<=float(row.feature_position)<=20:
        return ('POSITION_OPPORTUNITY','Assess content completeness and internal linking, subject to editorial review')
    return ('EDITORIAL_DIAGNOSIS','Human diagnosis needed; no specific page change is justified by score alone')

priority=holdout.copy()
priority['_rf_score']=heldout_model_score
priority['_rule_score']=heldout_rule_score
# Operational review queue follows the transparent rule. RF provides supporting risk evidence.
priority=priority.sort_values('_rule_score',ascending=False,kind='mergesort').copy()
priority['priority_rank']=np.arange(1,len(priority)+1)
priority['priority_band']=pd.cut(priority.priority_rank,[0,10,25,50,len(priority)+1],
                                  labels=['Priority 1–10','Priority 11–25','Priority 26–50','Outside top 50'],right=True)
top50=priority.head(50).copy()
reasons=[editorial_reason(row,train) for row in top50.itertuples(index=False)]
top50['reason_code']=[r for r,_ in reasons]
top50['suggested_review']=[a for _,a in reasons]
top50['evidence_level']=pd.cut(top50.feature_impressions,[-1,200,1000,float('inf')],
                                  labels=['limited','moderate','higher-volume'],right=False).astype(str)
REASON_COUNTS={str(k):int(v) for k,v in top50.reason_code.value_counts().items()}
EVIDENCE_COUNTS={str(k):int(v) for k,v in top50.evidence_level.value_counts().items()}
BAND_ROWS=[]
for start,end in ((1,10),(11,25),(26,50)):
    subset=top50[(top50.priority_rank>=start)&(top50.priority_rank<=end)]
    BAND_ROWS.append({'band':f'{start}–{end}','pages':int(len(subset)),
                      'slowdown':int((subset.reason_code=='VISIBLE_SLOWDOWN').sum()),
                      'monitor':int((subset.reason_code=='LOW_EVIDENCE').sum())})
print('Built local-only ranked queue (',len(priority),'observations).',sep='')
print('Only aggregated reason counts and priority-band summaries will be published.')
print('Top-50 reason distribution:',REASON_COUNTS)

## 5. Export only safe aggregate evidence

The notebook writes `work/reports/capstone_results_public.json` and four presentation-quality charts under `docs/assets/`. It never saves or downloads the modeling DataFrame or hashed-ID queue. The primary results table contains **only the fixed Random Forest versus the rule baseline**, satisfying the final paper’s requested comparison without publishing exploratory model-comparison metrics.

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from importlib.metadata import version as installed_version

# Aggregate publication record: excludes hashes, page-level labels, private queries and data rows.
REPORT={
    'study':'FlyRank ML Internship · Lane 2 · Refresh / Content Opportunity Scoring',
    'analysis_status':'measured_in_authorized_colab',
    'dataset':DATA_META,
    'observation_grain':'pseudonymized client × content page (one feature/outcome month-pair)',
    'eligible_rows':int(len(analysis)),
    'training_rows':int(len(train)),
    'holdout_rows':int(len(holdout)),
    'client_groups_total':int(analysis.client_id.nunique()),
    'training_client_groups':int(train.client_id.nunique()),
    'holdout_client_groups':int(holdout.client_id.nunique()),
    'client_group_overlap':int(len(set(train.client_id)&set(holdout.client_id))),
    'holdout_prevalence':float(holdout.label.mean()),
    'random_forest':RF_METRICS,
    'rule_baseline':RULE_METRICS,
    'rf_minus_baseline_bootstrap':UNCERTAINTY,
    'training_only_cv':{
         'folds':len(cv_records),
         'rf_precision_at_50_mean':float(np.mean([c['rf']['precision_at_50'] for c in cv_records])),
         'rule_precision_at_50_mean':float(np.mean([c['rule']['precision_at_50'] for c in cv_records])),
         'rf_average_precision_mean':float(np.mean([c['rf']['average_precision'] for c in cv_records])),
         'rule_average_precision_mean':float(np.mean([c['rule']['average_precision'] for c in cv_records])),
    },
    'secondary_later_window':LATER_RESULT,
    'fixed_random_forest':{'n_estimators':140,'max_depth':10,'min_samples_leaf':35,'max_features':'sqrt','random_state':SEED},
    'rule_weights':{'impression_exposure':0.40,'feature_month_slowdown':0.35,'below_position_peer_ctr':0.25},
    'feature_columns':NUMERIC+CATEGORICAL,
    'prohibited_columns':sorted(PROHIBITED),
    'top50_rule_reason_counts':REASON_COUNTS,
    'top50_rule_evidence_counts':EVIDENCE_COUNTS,
    'top50_rule_priority_bands':BAND_ROWS,
    'evaluation_status':('Historical April–May client-held-out period was inspected in development; '
                         'reported scores are descriptive and should not be called untouched confirmatory tests.'),
    'policy':('Rule score ranks the initial editorial review queue; Random Forest is an additional '
              'risk-ranking comparison. No intervention is triggered automatically.'),
    'created_utc':datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
}
assert REPORT['client_group_overlap']==0
assert not set(REPORT['feature_columns'])&PROHIBITED
assert REPORT['holdout_rows']>=100
# Detect stray identifiers in public aggregate object (keys only: values contain no raw rows).
json_text=json.dumps(REPORT,ensure_ascii=False,indent=2,allow_nan=False)
REPORT['runtime_packages'] = {p: installed_version(p) for p in ('duckdb','pandas','numpy','scikit-learn','matplotlib')}
(OUT/'work/reports/capstone_results_public.json').write_text(json_text,encoding='utf-8')

# Research-quality plots without hashes or record-level data.
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False})
ks=[10,20,50,100]
xs=np.arange(len(ks));w=.34
fig,ax=plt.subplots(figsize=(9.4,5.1))
ax.bar(xs-w/2,[RULE_METRICS[f'precision_at_{k}'] for k in ks],width=w,label='Rule-based benchmark',color='#2a6672')
ax.bar(xs+w/2,[RF_METRICS[f'precision_at_{k}'] for k in ks],width=w,label='Fixed Random Forest',color='#c07b54')
ax.axhline(RF_METRICS['prevalence'],ls='--',lw=1.25,color='#667085',label='Held-out prevalence')
ax.set_xticks(xs,[f'P@{k}' for k in ks]);ax.set_ylim(0,1.08)
ax.set_ylabel('Share of subsequently declining pages')
ax.set_title('Review precision at different editorial budgets',loc='left',fontweight='bold')
ax.legend(frameon=False,ncol=2);ax.grid(axis='y',alpha=.13)
fig.tight_layout();fig.savefig(OUT/'docs/assets/precision_at_k.png',dpi=180,bbox_inches='tight');plt.close(fig)

fig,ax=plt.subplots(figsize=(8.5,4.7))
metrics=['average_precision','roc_auc']
x=np.arange(len(metrics))
ax.bar(x-w/2,[RULE_METRICS[m] for m in metrics],width=w,label='Rule-based benchmark',color='#2a6672')
ax.bar(x+w/2,[RF_METRICS[m] for m in metrics],width=w,label='Fixed Random Forest',color='#c07b54')
ax.set_xticks(x,['Average Precision','ROC-AUC']);ax.set_ylim(0,1.08)
ax.set_title('Overall discrimination on the same held-out observations',loc='left',fontweight='bold')
ax.grid(axis='y',alpha=.13);ax.legend(frameon=False);fig.tight_layout()
fig.savefig(OUT/'docs/assets/overall_ranking.png',dpi=180,bbox_inches='tight');plt.close(fig)

reason_desc={'VISIBLE_SLOWDOWN':'Recent visibility slowdown','LOW_EVIDENCE':'Limited exposure',
             'BELOW_PEER_CTR':'Below-peer CTR','POSITION_OPPORTUNITY':'Position opportunity',
             'EDITORIAL_DIAGNOSIS':'General editorial diagnosis'}
items=sorted(REASON_COUNTS.items(),key=lambda kv:kv[1],reverse=True)
fig,ax=plt.subplots(figsize=(9.2,4.8))
labels=[reason_desc.get(k,k) for k,v in items][::-1]
counts=[v for k,v in items][::-1]
ax.barh(labels,counts,color='#2a6672')
for j,v in enumerate(counts):ax.text(v+0.25,j,str(v),va='center')
ax.set_xlim(0,max(counts+[1])*1.18)
ax.set_xlabel('Number of pages in the rule-ranked top 50')
ax.set_title('Diagnostic reasons attached to prioritized reviews',loc='left',fontweight='bold')
ax.grid(axis='x',alpha=.12);fig.tight_layout()
fig.savefig(OUT/'docs/assets/review_reasons.png',dpi=180,bbox_inches='tight');plt.close(fig)

# Fixed model’s training-derived impurity importance; associations only, not causal contributions.
feature_names=model.named_steps['preprocess'].get_feature_names_out()
importance=model.named_steps['model'].feature_importances_
idx=np.argsort(importance)[-9:]
fig,ax=plt.subplots(figsize=(9.1,5.0))
ax.barh([feature_names[i].replace('num__','').replace('cat__position_tier_','tier: ') for i in idx],importance[idx],color='#c07b54')
ax.set_title('Random Forest feature importance (training-derived)',loc='left',fontweight='bold')
ax.set_xlabel('Mean decrease in impurity · descriptive, not causal')
ax.grid(axis='x',alpha=.12);fig.tight_layout()
fig.savefig(OUT/'docs/assets/feature_importance.png',dpi=180,bbox_inches='tight');plt.close(fig)
print('Exported one aggregate JSON report and four non-identifying figures.')

## 6. Write the eight-section FlyRank capstone report

The report mirrors the official eight-section assessment template, and its numerical values come from the same aggregate `REPORT` used for the paper. This is a public-safe Markdown artifact.


In [ ]:
"""Public-safe rubric report builder. Uses aggregate metrics only, never raw warehouse data."""
from pathlib import Path
import datetime
import json
import importlib.metadata as md

REPO_URL = 'https://github.com/pretom26/ml_internship_flyrankAI'
AUTHOR_NAME = 'Pretom Areefin Pranto'
AUTHOR_ROLE = 'Machine Learning Intern — FlyRank AI'

def metrics(v):
    return f'{v:.3f}'

def render_report(d, name=AUTHOR_NAME, role=AUTHOR_ROLE, repo=REPO_URL, report_date='2026-10-21', versions=None):
    ds = d['dataset']
    rf = d['random_forest']
    base = d['rule_baseline']
    later = d['secondary_later_window']
    prev = d['holdout_prevalence']
    majority = max(prev, 1-prev)
    seed=d['fixed_random_forest']['random_state']
    def n(x): return f'{x:,}'
    def score(x): return f'{x:.3f}'
    def delta(x): return f'{x:+.3f}'
    fmt_versions = '\n'.join(f'  - `{p}`: `{v}`' for p,v in (versions or {}).items()) if versions else (
        '  - The completed aggregate export does not record exact runtime package versions; '
        'the runnable notebook declares DuckDB >=1.4,<2, pandas >=2.0, numpy >=1.24, scikit-learn >=1.3, matplotlib >=3.7. '
        'These ranges permit repeat execution, although they are not a byte-for-byte frozen environment.'
    )
    interval=d['rf_minus_baseline_bootstrap']
    def ci(which):
        z=interval[which]
        return f"{delta(z['delta'])} [{delta(z['low_95'])}, {delta(z['high_95'])}]"
    features=', '.join('`'+f+'`' for f in d['feature_columns'])
    v= f'''# Capstone Report — Prioritizing Content Reviews with Search Performance Signals

- **Author:** {name} — {role}
- **Lane:** Lane 2 · Refresh / Content Opportunity Scoring
- **Repo:** {repo}
- **Date:** {report_date}
- **Research paper:** `docs/index.html` (GitHub Pages); the verified live URL is recorded separately in `submission/paper_url.txt` at publication.
- **Results source:** `work/reports/capstone_results_public.json`, generated from the gated warehouse in the authorized Colab analysis.

## 1. Problem framing

An editorial team has limited time and must decide which existing content pages warrant a diagnostic review first. The **unit of analysis** is one pseudonymized client × content page, aggregated over April 2026, with an observed outcome in May 2026. The outputs are a **priority score**, a **ranked review queue**, a **reason code**, and an **editorial review action**. A reviewer should inspect the underlying page's tracking quality, indexing, search-demand changes, intent match, and content currency before making any revisions.

False positives waste scarce editorial review time and may induce unnecessary content changes; false negatives cause truly declining pages to receive attention later. Review itself is not a refresh treatment. A baseline allows users to inspect how recommendations are made, while the Random Forest tests whether interactions among observable prior-month signals improve evidence-based prioritization. **Success is defined as ranking subsequently declining pages high enough to be reviewed under a limited editorial budget, not proving any benefit from refreshes.**

## 2. Data safety

**Approved release:** {ds['release']}; gated monthly Parquet partitions of `fact_content_daily_performance` loaded through DuckDB's Hugging Face access inside authorized Google Colab. April 1–30, 2026 supplies features; May 1–31, 2026 supplies subsequent outcomes. The cohort includes **{n(d['eligible_rows'])} eligible observations across {d['client_groups_total']} client groups**; training uses {n(d['training_rows'])} records from {d['training_client_groups']} clients and evaluation uses {n(d['holdout_rows'])} records from {d['holdout_client_groups']} different clients. The observed client overlap is **{d['client_group_overlap']}**.

Eligibility requires at least 100 April impressions and at least five days with GSC impression observations **in both months**. Exclusions include absent following-month records, inadequate coverage, low April exposure, or missing keys/labels. Duplicate client–page combinations cause an explicit error. A missing next-month row is **not** counted as zero impressions. Outcome-month coverage is used for **retrospective sample eligibility only** and is **not an input feature**; it means the measured cohort may not represent all pages encountered in production.

The model **does not use** client/content hashes, URLs, names, domains, titles, keywords, private queries, the outcome-period impressions or observed-day counts, the binary outcome, or starter-dataset label-derived columns such as `trend_direction` and `trend_pct`. Likewise, FlyRank's product-derived decision flags (such as `health_score` or `needs_ctr_fix`) are not used. IDs are held in memory solely for joining, validating unique page grain, and grouping the split. The exported JSON and figures contain **aggregate** evidence, not row-level private exports.

**Publication boundary:** this capstone exports only numerical summaries and figures; its private page-level ranking stays in the authorized analysis session. All other repository material, including historical notebook outputs, must follow the same data-use rules. The artifact-level privacy check is not a retrospective audit of the repository’s commit history.

## 3. Baseline

The transparent benchmark assigns a descending editorial priority based on **40%** capped log impressions, **35%** negative intra-April momentum, and **25%** click-through-rate shortfall relative to **training-client-only** position-tier peers. The exposure cap and peer CTR summaries are derived from training clients, not held-out labels. This rule does not use the following-month outcome. It is a fair comparator because it uses the same eligible held-out pages, feature window, target labels, and ranking metrics as the Random Forest.

**Observed baseline:** Precision@10 = {score(base['precision_at_10'])}; Precision@20 = {score(base['precision_at_20'])}; **Precision@50 = {score(base['precision_at_50'])}**; Precision@100 = {score(base['precision_at_100'])}; Average Precision = {score(base['average_precision'])}; ROC-AUC = {score(base['roc_auc'])}. The baseline’s top-50 list contains **{int(round(50*base['precision_at_50']))} observed declines** and **{50-int(round(50*base['precision_at_50']))} false-positive priorities** under the research label.

## 4. Model / analysis

**Final model:** one fixed scikit-learn `RandomForestClassifier` with 140 trees, `max_depth=10`, `min_samples_leaf=35`, `max_features='sqrt'`, `n_jobs=-1`, and `random_state={seed}`. The exact settings were informed by earlier development; this final report does not include or claim new experimental model comparisons. A training-only preprocessing pipeline imputes numeric missing values, scales numeric features, and one-hot encodes the position tier. Training uses only the designated training clients.

**Exact model features ({len(d['feature_columns'])}):** {features}. All derive from the **feature month**, including April's early-versus-late impression momentum and interactions with exposure. Raw early/late impression aggregates are used **only** to construct April momentum, not as additional unreviewed model columns.

**One-sentence target:** A page is labeled 1 when its May 2026 impressions per calendar day are **strictly more than 30% lower** than its April 2026 impressions per calendar day, conditional on the observation criteria. This is an observed next-month decline proxy, not an indication that a page should automatically be rewritten.

## 5. Evaluation

A fixed `GroupShuffleSplit(test_size=0.2, random_state={seed})` holds out entire client groups; the resulting unequal client sizes create a {d['holdout_rows']/d['eligible_rows']:.1%} row-level holdout despite 20% of the **groups** being reserved. Three training-only grouped validation folds check consistency. The same {n(d['holdout_rows'])} held-out rows and outcome labels are scored by both the rule and the Random Forest. The April-to-May label starts strictly after the feature window; future metrics and any transformed outcome are excluded from predictors. The held-out April–May group was previously examined during development, so these scores are **descriptive rather than a newly untouched confirmation test**.

**Base rate:** The positive outcome prevalence is **{prev:.2%}** ({int(round(prev*d['holdout_rows'])):,} positives among {n(d['holdout_rows'])} held-out page observations, rounded from aggregate reporting). The majority-class fraction is **{majority:.2%}**, which is not a ranking quality measure. Precision@K should therefore be read alongside the prevalence, lift, Average Precision, and ROC-AUC.

| Same historical held-out cohort | Rule benchmark | Random Forest |
|:--|--:|--:|
| Precision@10 | {base['precision_at_10']:.3f} | {rf['precision_at_10']:.3f} |
| Precision@20 | {base['precision_at_20']:.3f} | {rf['precision_at_20']:.3f} |
| **Precision@50** | **{base['precision_at_50']:.3f}** | **{rf['precision_at_50']:.3f}** |
| Precision@100 | {base['precision_at_100']:.3f} | {rf['precision_at_100']:.3f} |
| Average Precision | {base['average_precision']:.3f} | {rf['average_precision']:.3f} |
| ROC-AUC | {base['roc_auc']:.3f} | {rf['roc_auc']:.3f} |
| Lift@50 over {prev:.2%} prevalence | {base['lift_at_50']:.2f}× | {rf['lift_at_50']:.2f}× |

**Budgeted error analysis:** Among the top 50 recommendations, the rule prioritizes {int(round(50*base['precision_at_50']))} positives and {50-int(round(50*base['precision_at_50']))} negatives, while the Random Forest prioritizes {int(round(50*rf['precision_at_50']))} positives and {50-int(round(50*rf['precision_at_50']))} negatives. Both reach {base['precision_at_100']:.0%} Precision@100. These observations show why a higher global ROC-AUC does **not** guarantee a better 50-page queue. The top-K false positives require manual diagnosis; no public page records are disclosed.

**Descriptive paired client-bootstrap intervals (Random Forest − rule; 400 repetitions):** P@50 {ci('precision_at_50')}; Average Precision {ci('average_precision')}; ROC-AUC {ci('roc_auc')}. The difference intervals for P@50 and Average Precision include zero, and only {d['holdout_client_groups']} held-out clients were available, so the intervals should not be over-interpreted as definitive statistical proof.

**Secondary time-window check:** May predictors → June outcomes for {n(later['n_rows'])} observations across {later['n_groups']} client groups, using the **unchanged** model without refitting. P@50: rule {later['rule_metrics']['precision_at_50']:.3f} versus Random Forest {later['rf_metrics']['precision_at_50']:.3f}; Average Precision: rule {later['rule_metrics']['average_precision']:.3f} versus Random Forest {later['rf_metrics']['average_precision']:.3f}. The same client holdout is reused and May features are contemporaneous with the original training-label month for other clients, so this is a **robustness check**, not a clean prospective trial.

## 6. Interpretation

The main negative result matters: at a fixed budget of 50 human reviews, the **transparent rule is better** ({base['precision_at_50']:.0%} versus {rf['precision_at_50']:.0%}), even though Random Forest ranks more effectively across the full evaluation population (Average Precision {rf['average_precision']:.3f} versus {base['average_precision']:.3f}; ROC-AUC {rf['roc_auc']:.3f} versus {base['roc_auc']:.3f}). There is no universal winner; the choice depends on whether editors prioritize a small high-confidence queue or broader risk discovery.

The notebook's training-derived impurity feature-importance figure places **decline × log exposure**, **within-month momentum**, and **downward momentum** among the strongest contributors, followed by position and CTR-related signals. These variables are correlated transformations of observed search data; impurity importance is **not** a causal explanation and can split importance among related variables. This analysis does not show that changing any single predictor would change rankings or prevent a decline.

![Figure: Precision at fixed review budgets](../docs/assets/precision_at_k.png)

![Figure: Training-derived Random Forest predictor importance](../docs/assets/feature_importance.png)

## 7. Recommendation

Use the **rule-based ranking as the initial operational queue** because it performs better at P@50 in both the historical comparison and secondary time-window check. Treat the Random Forest score as supporting diagnostic evidence rather than a trigger for a content refresh. An authorized editor can inspect the highest-ranked pages **within the private warehouse environment**; no page-specific recommendations are published publicly.

| Priority band | Pages | Evidence-guided human action |
|:--|--:|:--|
| 1–10 | 10 | Verify tracking, index coverage, demand patterns and content currency immediately; avoid automatic rewriting. |
| 11–25 | 15 | Review the next 15 after excluding measurement anomalies and seasonal effects. |
| 26–50 | 25 | Maintain a monitored watchlist and prioritize if new observations strengthen the concern. |

**Observed rule-ranked top-50 reasons:** {', '.join(f'`{k}`: {v}' for k,v in d['top50_rule_reason_counts'].items())}. **Evidence level:** {', '.join(f'`{k}`: {v}' for k,v in d['top50_rule_evidence_counts'].items())}. In particular, 19 records with low evidence merit monitoring before invasive edits. The reason codes are review prompts—not validated diagnoses or a claim that a recommended edit improves organic visibility.

**Confidence and cost:** A top-50 positive rate does not imply certain intervention benefit or calibrated probability. Editorial effort should be proportional to exposure, measurement reliability, and the cost of a wrong call. Flagging a page for audit should not automatically prune, merge, or rewrite it.

## 8. Reproducibility

**Repository:** {repo}. Main notebook: [`work/notebooks/capstone.ipynb`]({repo}/blob/main/work/notebooks/capstone.ipynb). Numerical reference: [`work/reports/capstone_results_public.json`]({repo}/blob/main/work/reports/capstone_results_public.json). Public paper: `docs/index.html` under GitHub Pages. This report is an additional required artifact at `work/capstone_report.md`.

**Exact Colab workflow (from a fresh repo checkout):**

```bash
git clone {repo}.git
cd ml_internship_flyrankAI
# In Google Colab: File → Open notebook → GitHub; select
# work/notebooks/capstone.ipynb from this repository.
# In Colab Secrets, grant this notebook access to HF_TOKEN (read token).
# Runtime → Run all, then review generated reports and site artifacts.
```

The notebook's first setup cell runs:

```python
%pip -q install "duckdb>=1.4,<2" "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.3" "matplotlib>=3.7"
```

**Seeds:** Outer `GroupShuffleSplit` and Random Forest: `{seed}`. Internal grouped validation split: `2026`. Paired client-bootstrap resampling: `2026` (400 replicates). The reported feature/construction logic and charts live inside the notebook, with no additional helper modules required. Queries need authorized access to the **gated** FlyRank warehouse; no raw training data or secret keys are committed or bundled. A repeat execution may be subject to remote release stability, environment changes, or deterministic tie ordering.

**Environment specification:**
{fmt_versions}

**Generate publication materials:** the final notebook writes the report, HTML, four charts and aggregate JSON in one authorized Colab run. To regenerate only the eight-section Markdown report from the saved aggregate JSON, run `python work/generate_capstone_report.py` at the repository root. This operation never reads raw warehouse records.

**Verification:** a reproducible run verifies zero client overlap, equal evaluation cohorts, regenerated aggregate statistics and renderable figures. Run `python work/validate_release.py` at the repository root after copying the Colab outputs. Its checks compare the public metrics, report, HTML and chart paths; they cannot replace an actual authorized rerun of the gated data.

---

### Claims and data-use statement

This study measures later impressions decline among eligible pages; it does not test whether editing pages improves future search performance, recover any search engine’s ranking algorithm, or validate page-level interventions. The outcome base rate and adverse benchmark comparisons are shown alongside the favorable ML results. Published materials consist of aggregate statistics and figures; authorized raw records and private ranking queues remain outside the repository. The direct publication URL belongs in `submission/paper_url.txt` only after the live website has been verified.
'''
    return v


# Build the exact rubric-aligned report from the same in-memory aggregate results as the paper.
report_md = render_report(REPORT, name=AUTHOR['full_name'], role=AUTHOR['position'], repo=REPO_URL, report_date='2026-10-21', versions=REPORT.get('runtime_packages'))
(OUT/'work/capstone_report.md').write_text(report_md, encoding='utf-8')
print('Report:', OUT/'work/capstone_report.md')


## 7. Render the final research paper

The HTML report draws results from the same aggregate `REPORT` as the Markdown assessment and includes the five-sentence abstract, required research sections, author profile, charts, and data acknowledgment.


In [ ]:
from string import Template
from urllib.parse import urlparse

# Human-editable final paper: edit AUTHOR and PAPER_EDIT at the TOP of this notebook,
# or manually edit the generated docs/index.html after running this cell.
# Manual HTML edits will be overwritten if the notebook is rerun.

def safe_url(u):
    u=(u or '').strip()
    p=urlparse(u)
    return u if p.scheme=='https' and bool(p.netloc) else ''

def escaped(value):
    return html.escape(str(value),quote=True)

def build_paper(report=None,output_dir=OUT):
    name=AUTHOR['full_name'].strip()
    bio=AUTHOR['bio'].strip()
    if report is None:
        raise ValueError('A measured aggregate result is required to create the paper.')
    if not name or not bio:
        raise ValueError('A public author name and biography are required.')
    status='Research paper · measured results'
    title=escaped(PAPER_EDIT['title'])
    subtitle=escaped(PAPER_EDIT['subtitle'])
    counts=''
    figure_html=''
    summary_rows=''
    bootstrap_html=''
    importance_fig=''
    later_html=''
    reasons_html=''
    bands_html=''
    feattext=', '.join(NUMERIC[:6]+NUMERIC[6:]+CATEGORICAL)
    if report is not None:
        summary_rows=''
        bootstrap_html=''
        rf=report['random_forest'];base=report['rule_baseline']
        winners=('The transparent benchmark was more precise among the top 50 recommendations' if base['precision_at_50']>rf['precision_at_50']
                 else 'The Random Forest was at least as precise among the top 50 recommendations')
        abstract=(
         'This study asks whether search-performance signals can prioritize content pages for investigation before substantial visibility decline. '
         'We constructed a page-level dataset from consecutive monthly partitions of the FlyRank internship warehouse and defined a decline using the following month’s impressions per calendar day. '
         'A fixed Random Forest and a transparent rule-based score were evaluated on the same previously examined, client-group-held-out observations. '
         f"The Random Forest achieved {rf['precision_at_50']:.0%} Precision@50 and {rf['average_precision']:.3f} Average Precision, compared with {base['precision_at_50']:.0%} and {base['average_precision']:.3f} for the benchmark. "
         f'{winners}, demonstrating that editorial review budgets and broader ranking quality can favor different methods.'
        )
        counts=(f"{report['eligible_rows']:,} eligible observations across {report['client_groups_total']} client groups; "
                f"{report['training_rows']:,} training and {report['holdout_rows']:,} held-out observations "
                f"({report['holdout_client_groups']} previously unseen clients).")
        measure_labels={'precision_at_10':'Precision@10','precision_at_20':'Precision@20',
                        'precision_at_50':'Precision@50','precision_at_100':'Precision@100',
                        'average_precision':'Average Precision','roc_auc':'ROC-AUC'}
        for field,label in measure_labels.items():
            summary_rows+=(f'<tr><th scope="row">{escaped(label)}</th><td>{base[field]:.3f}</td>'
                           f'<td>{rf[field]:.3f}</td></tr>')
        figure_html='''<div class="figures">
          <figure><img src="assets/precision_at_k.png" alt="Grouped bars comparing rule baseline and Random Forest Precision at 10, 20, 50 and 100." loading="lazy"><figcaption>Figure 1. Review precision at fixed editorial budgets; dashed line indicates held-out outcome prevalence.</figcaption></figure>
          <figure><img src="assets/overall_ranking.png" alt="Bars comparing Average Precision and ROC-AUC for the two methods." loading="lazy"><figcaption>Figure 2. Overall ranked discrimination on identical held-out observations.</figcaption></figure>
          </div>'''
        for metric, values in report['rf_minus_baseline_bootstrap'].items():
            bootstrap_html+=(f'<tr><th scope="row">{escaped(measure_labels.get(metric,metric))}</th>'
                             f"<td>{values['delta']:+.3f}</td><td>[{values['low_95']:+.3f}, {values['high_95']:+.3f}]</td></tr>")
        later=report['secondary_later_window']
        if later['status']=='evaluated':
            a=later['rule_metrics'];b=later['rf_metrics']
            later_html=(f'<p><strong>Later-window robustness (secondary):</strong> With May features and June outcomes, '
                        f"the unchanged model was evaluated on {later['n_rows']:,} observations across "
                        f"{later['n_groups']} originally held-out clients. "
                        f"Precision@50 was {a['precision_at_50']:.3f} for the rule and "
                        f"{b['precision_at_50']:.3f} for Random Forest; "
                        f"Average Precision was {a['average_precision']:.3f} and {b['average_precision']:.3f}, respectively. "
                        'The evaluation reuses client groups and overlapping calendar-month information, '
                        'so it is a sensitivity check rather than a prospective independent trial.</p>')
        else:
            later_html=f'<p><strong>Later-window robustness:</strong> {escaped(later.get("note","Not available"))}</p>'
        desc={'VISIBLE_SLOWDOWN':('Visibility slowdown','Examine demand, indexing and topic currency before editing'),
              'LOW_EVIDENCE':('Limited exposure','Monitor rather than initiate an expensive rewrite'),
              'BELOW_PEER_CTR':('Below-peer CTR','Review metadata relevance and search intent'),
              'POSITION_OPPORTUNITY':('Position opportunity','Audit usefulness and internal linking'),
              'EDITORIAL_DIAGNOSIS':('General diagnosis','Request human editorial review before any action')}
        importance_fig='<figure><img src="assets/feature_importance.png" alt="Fixed Random Forest training-derived feature importances" loading="lazy"><figcaption>Figure 4. Training-derived feature importance; not evidence of causality.</figcaption></figure>'
        reason_rows=''
        for k,v in sorted(report['top50_rule_reason_counts'].items(),key=lambda kv:-kv[1]):
            head,action=desc.get(k,(k,'Human review'))
            reason_rows+=f'<tr><th scope="row">{escaped(head)}</th><td>{v}</td><td>{escaped(action)}</td></tr>'
        reasons_html=(f'<p>The initial review queue is ranked using the prespecified transparent score. '
                      f'Random Forest is reported as a complementary learned risk-ranking method. '
                      f'No client, page or query identifiers are published.</p><div class="tablewrap"><table>'
                      f'<thead><tr><th>Review evidence</th><th>Top-50 pages</th><th>Recommended diagnostic action</th></tr></thead>'
                      f'<tbody>{reason_rows}</tbody></table></div>'
                      '<figure><img src="assets/review_reasons.png" alt="Aggregate counts of editorial review reason codes among the rule-ranked top 50." loading="lazy"><figcaption>Figure 3. Aggregated review reasons for the rule-ranked top 50; not an intervention outcome.</figcaption></figure>')
        for band in report['top50_rule_priority_bands']:
            bands_html+=(f"<tr><th scope='row'>{escaped(band['band'])}</th><td>{band['pages']}</td>"
                         f"<td>{band['slowdown']}</td><td>{band['monitor']}</td></tr>")
    author_full=escaped(name)
    affiliation=(' · '+escaped(AUTHOR['affiliation'])) if AUTHOR['affiliation'].strip() else ''
    links=''
    for key,shown in [('linkedin','LinkedIn'),('portfolio','Portfolio')]:
        url=safe_url(AUTHOR.get(key,''))
        if url: links+=f'<a href="{escaped(url)}" rel="noopener noreferrer">{shown} ↗</a>'
    email=AUTHOR.get('email','').strip()
    if email and re.fullmatch(r'[^@\s]+@[^@\s]+\.[^@\s]+',email):
        links+=f'<a href="mailto:{escaped(email)}">Email ↗</a>'
    # Source links are public and verified; no source data records are embedded in HTML.
    repo=safe_url(REPO_URL)
    initials=''.join(part[0].upper() for part in re.findall(r'[A-Za-z]+',name)[:2]) or 'AI'
    release=report['dataset']['release'] if report else 'FlyRank internship warehouse v20260703'
    if report:
        data=report['dataset']
        dates=f"{data['feature_start']} to {data['feature_end_exclusive']} (exclusive) for features; {data['outcome_start']} to {data['outcome_end_exclusive']} (exclusive) for outcomes"
        eligibility=data['eligibility']
    else:
        dates='April 2026 features, May 2026 outcomes (adjacent non-overlapping months)'
        eligibility='At least 100 feature-month impressions and five observed GSC days in each month'
    page=Template(r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>$TITLE · FlyRank Lane 2 Research</title><meta name="description" content="Group-validated observational research on ranking content pages for editorial review.">
<style>
:root{--ink:#18313d;--muted:#526874;--teal:#285f6c;--warm:#a56e4c;--line:#dbe4e7;--paper:#fbfcfc;--soft:#f0f5f5;--max:1030px}
*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:var(--paper);color:var(--ink);font:16px/1.72 system-ui,-apple-system,Segoe UI,Arial,sans-serif}a{color:#215c68;text-underline-offset:3px}a:hover{color:#8b5735}p{margin:0 0 1.15rem}h1,h2,h3{line-height:1.18;letter-spacing:-.025em}h1,h2{font-family:Georgia,'Times New Roman',serif}h1{font-size:clamp(2.5rem,5.2vw,4.5rem);max-width:960px;margin:24px 0}h2{font-size:clamp(1.75rem,3.5vw,2.7rem);margin:0 0 25px}h3{font-size:1.12rem;margin:0 0 12px}.shell{width:min(var(--max),calc(100% - 44px));margin:auto}.hero{background:linear-gradient(132deg,#18313d,#285c67);color:#fff;padding:84px 0 70px}.hero .lead{font-size:1.25rem;max-width:730px;color:#dbeaec}.eyebrow{font-size:.73rem;font-weight:800;letter-spacing:.16em;text-transform:uppercase;color:#c1d9dc}.status{display:inline-block;padding:7px 12px;background:#ffffff16;border:1px solid #ffffff55;border-radius:5px;color:#eef8f8;font-size:.85rem}.herofooter{margin-top:30px;color:#d9e8ea;font-size:.91rem}.sticky{position:sticky;top:0;background:#ffffffed;z-index:10;border-bottom:1px solid var(--line);backdrop-filter:blur(9px)}.navigation{display:flex;gap:25px;align-items:center;overflow:auto;white-space:nowrap;padding:15px 0;font-size:.84rem}.navigation a{text-decoration:none;color:var(--ink)}.navigation a:hover{text-decoration:underline}main section{padding:74px 0;border-bottom:1px solid var(--line);scroll-margin-top:68px}.sectionnum{font-size:.75rem;color:var(--teal);font-weight:800;letter-spacing:.11em;text-transform:uppercase;margin-bottom:12px}.lede{font:1.2rem/1.65 Georgia,serif;color:#304855}.two{display:grid;grid-template-columns:1fr 1fr;gap:26px}.panel{background:white;border:1px solid var(--line);padding:25px;border-radius:10px}.panel p:last-child{margin-bottom:0}.facts{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin:27px 0}.fact{padding:19px 22px;background:#edf3f3;border-top:3px solid #37747d}.fact strong{font-size:1.55rem;display:block;color:#234f5a}.fact small{font-size:.76rem;color:var(--muted)}.tablewrap{overflow:auto;border:1px solid var(--line);border-radius:8px;margin:22px 0}table{border-collapse:collapse;width:100%;background:white;font-size:.93rem}th,td{padding:13px 15px;border-bottom:1px solid var(--line);text-align:left}th{font-weight:650}thead th{background:#eaf2f3}tbody tr:last-child th,tbody tr:last-child td{border-bottom:none}.figures{display:grid;grid-template-columns:1fr 1fr;gap:20px}figure{margin:25px 0;background:#fff;border:1px solid var(--line);border-radius:9px;padding:16px}figure img{display:block;width:100%;height:auto}figcaption{color:var(--muted);font-size:.82rem;margin-top:12px}.callout{background:#eef4f4;border-left:4px solid #397380;padding:19px 23px;margin:20px 0}.muted{color:var(--muted)}.small{font-size:.88rem}.ident{display:flex;gap:25px;align-items:flex-start}.avatar{width:92px;height:92px;border-radius:50%;background:#e0ebed;display:grid;place-items:center;font:2rem Georgia,serif;color:var(--teal);flex-shrink:0}.links{display:flex;flex-wrap:wrap;gap:16px}footer{background:#18313d;color:#e2edef;padding:35px 0;font-size:.85rem}footer a{color:#e2edef}code{font-size:.87em;overflow-wrap:anywhere;background:#eaf1f1;padding:2px 5px;border-radius:3px}ul{padding-left:21px}li{margin-bottom:9px}@media(max-width:700px){.hero{padding:59px 0}.two,.figures,.facts{grid-template-columns:1fr}main section{padding:55px 0}.ident{flex-direction:column}.navigation{gap:17px}}@media print{.sticky{display:none}.hero{background:#fff;color:#172c34;padding:35px 0}.hero .lead,.herofooter{color:#172c34}main section{padding:30px 0}a{text-decoration:none}figure{break-inside:avoid}}
</style></head><body>
<header class="hero"><div class="shell"><div class="eyebrow">FlyRank AI · Machine Learning Internship · Lane 02 / Research</div><h1>$TITLE</h1><p class="lead">$SUBTITLE</p><span class="status">$STATUS</span><div class="herofooter">$AUTHOR_NAME · $AUTHOR_POSITION</div></div></header>
<nav class="sticky" aria-label="Paper sections"><div class="shell navigation"><a href="#abstract">Abstract</a><a href="#introduction">Introduction</a><a href="#data">Data</a><a href="#methods">Methods</a><a href="#findings">Results</a><a href="#limitations">Limitations</a><a href="#recommendations">Recommendations</a><a href="#reproducibility">Reproducibility</a><a href="#author">Author</a><a href="#credit">Credit</a></div></nav>
<main class="shell">
<section id="abstract"><div class="sectionnum">Abstract · five sentences</div><h2>Prioritizing evidence, not promising traffic</h2><p class="lede">$ABSTRACT</p></section>
<section id="introduction"><div class="sectionnum">01 / Introduction &amp; problem statement</div><h2>The editorial prioritization problem</h2><p>$PROBLEM</p><p>$CONTRIBUTION</p><div class="callout"><strong>Decision supported:</strong> which existing pages should be audited first, and why. The model is not intended to decide that a page must be rewritten, merged, deleted or refreshed automatically.</div></section>
<section id="data"><div class="sectionnum">02 / Data and eligibility</div><h2>A restricted research warehouse, presented responsibly</h2><div class="facts"><div class="fact"><small>Release</small><strong>v20260703</strong></div><div class="fact"><small>Feature window</small><strong>April 2026</strong></div><div class="fact"><small>Primary label window</small><strong>May 2026</strong></div></div><p><strong>Source:</strong> <a href="https://huggingface.co/datasets/FlyRank/internship-warehouse">FlyRank internship warehouse</a> ($RELEASE). <strong>Analyzed table:</strong> <code>fact_content_daily_performance</code>; each SQL aggregation groups a pseudonymized client–content pair. The remaining dimension and query-level tables were not required for this scoped study.</p><p><strong>Dates:</strong> $DATES. <strong>Eligibility:</strong> $ELIGIBILITY. Pairs with inadequate outcome-month coverage are excluded, not interpreted as having zero impressions; this avoids invented declines but may cause selection bias.</p><p><strong>Study population:</strong> $COUNTS</p><p class="small muted">No client names, domains, URLs, keywords, private queries, access tokens, or raw/page-level data exports are embedded in this site.</p></section>
<section id="methods"><div class="sectionnum">03 / Methodology</div><h2>One fixed model and one transparent benchmark</h2><div class="two"><div class="panel"><h3>Outcome and feature definition</h3><p>Decline is an impressions-per-calendar-day reduction <strong>greater than 30%</strong> between the feature month and the next month. Exposure below 100 prior-month impressions and sparse GSC observation are excluded.</p><p>Only feature-month exposure, clicks, CTR, weighted position, observation coverage, within-month momentum and their deterministic transformations enter the model.</p></div><div class="panel"><h3>Split and leakage audit</h3><p>A fixed GroupShuffleSplit holds out 20% of client groups by count; row fractions differ because client histories are unequal. The model is fitted on training clients only. The same held-out pages and labels evaluate the model and benchmark.</p><p>Month-of-outcome metrics, hashed identifiers, target-period observation counts and label-derived flags are excluded from model predictors.</p></div></div><p><strong>Prespecified benchmark:</strong> 40% relative impression exposure + 35% within-feature-month deceleration + 25% below-position-peer CTR. The reference distribution is derived from the training clients only.</p><p><strong>Fixed ML model:</strong> Random Forest (140 trees, maximum depth 10, minimum leaf size 35, square-root feature subsampling, random seed 42). Median imputation and categorical encoding are fitted inside the training pipeline. Features include log-exposure, per-observed-day impressions, downward momentum, and exposure interactions.</p><p><strong>Validation:</strong> training-only client-group checks, one predeclared client-held-out comparison, descriptive paired client-cluster bootstrap intervals, and—when data coverage permits—a later-outcome robustness check on the same held-out clients.</p></section>
<section id="findings"><div class="sectionnum">04 / Results</div><h2>Model and benchmark on identical pages</h2>$FIGURES<div class="tablewrap"><table><thead><tr><th>Metric</th><th>Rule benchmark</th><th>Random Forest</th></tr></thead><tbody>$METRIC_ROWS</tbody></table></div><p class="small muted">Precision@K measures the share of observed subsequent declines in the first K reviewed pages. Average Precision summarizes precision–recall ranking; ROC-AUC summarizes pairwise discrimination. All refer to the same primary held-out sample.</p><p><strong>Observed held-out outcome base rate:</strong> $BASE_RATE; <strong>majority class:</strong> $MAJORITY_RATE. At a 50-page editorial budget, lift relative to this prevalence is $RULE_LIFT× for the rule and $RF_LIFT× for Random Forest. The rule ranks $RULE_TP observed declines and $RULE_FP non-declining pages among its top 50; Random Forest ranks $RF_TP and $RF_FP, respectively.</p><h3>Uncertainty: Random Forest minus benchmark</h3><div class="tablewrap"><table><thead><tr><th>Metric</th><th>Observed difference</th><th>Descriptive 95% cluster-bootstrap interval</th></tr></thead><tbody>$BOOT_ROWS</tbody></table></div><p class="small muted">Bootstrap samples resample complete held-out client groups, not individual rows; only a limited number of groups are available. The intervals should not be overinterpreted as definitive significance tests.</p>$LATER<div class="two"><div class="panel"><h3>Interpretation</h3><p>A stronger overall ranking statistic does not guarantee a better top-50 review queue. The two methods are compared openly, including instances where the rule benchmark is preferable under a limited editorial budget.</p></div><div class="panel"><h3>Predictor contributions</h3>$IMPORTANCE_FIG</div></div></section>
<section id="limitations"><div class="sectionnum">05 / Limitations and research integrity</div><h2>What these findings can—and cannot—support</h2><p>This is an observational decision-support study using monthly aggregated search performance. The label captures later impressions decline, <em>not</em> whether a content refresh would improve results. The study does not reverse-engineer Google’s ranking system or establish that edits cause gains.</p><p>Search demand, seasonality, client-level reporting gaps, shifting query mix, measurement quality and the selection of pages with sufficient data can affect the results. The decline target is based on prior impressions in its denominator, making exposure a potentially mechanically related predictor. Model scores are not calibrated probabilities. A limited number of test client groups means uncertainty intervals may be unstable.</p><p><strong>Evaluation-history disclosure:</strong> The April–May held-out cohort was inspected during earlier method development. It is not presented as a newly untouched confirmatory test. Model settings are fixed for this final presentation, and the benchmark is measured on exactly the same held-out cohort.</p><p>The later-window check, when available, reuses held-out groups and some calendar-month information; it is descriptive robustness evidence rather than a fully independent prospective validation.</p></section>
<section id="recommendations"><div class="sectionnum">06 / Ranked editorial recommendations</div><h2>From ranking scores to concrete review actions</h2><p>The operational queue uses a transparent score to order pages for human review; the Random Forest provides an additional risk assessment and evaluation comparison. For publication, only grouped priority bands and reason-code frequencies are shown. The page-level queue remains inside the authorized Colab session.</p><div class="tablewrap"><table><thead><tr><th>Priority ranks</th><th>Pages</th><th>With visibility slowdown</th><th>Limited evidence</th></tr></thead><tbody>$BAND_ROWS</tbody></table></div>$REASONS<div class="callout"><strong>Action playbook:</strong> (1) verify tracking, seasonality and indexing before acting; (2) inspect search-intent alignment and content currency for slowing pages; (3) review titles and descriptions only where CTR is meaningfully below position peers; (4) monitor thin-evidence cases; (5) document human decisions and evaluate intervention outcomes separately.</div></section>
<section id="reproducibility"><div class="sectionnum">07 / Reproducibility</div><h2>A public method with protected observations</h2><p>The full analysis is in <a href="$REPO/blob/main/work/notebooks/capstone.ipynb">the self-contained capstone notebook</a>, run in Google Colab. It authenticates with a read-only <code>HF_TOKEN</code> from Colab Secrets, aggregates month-partitioned Parquet via DuckDB, and performs all modeling inside the notebook.</p><p>Downloadable public outputs are limited to this paper, aggregate charts and <a href="$REPO/blob/main/work/reports/capstone_results_public.json">the aggregate metrics JSON</a>, and the <a href="$REPO/blob/main/work/capstone_report.md">eight-section assessment report</a>. The <a href="$REPO">public GitHub repository</a> contains the notebook and previous weekly work without publishing raw data. Access to the gated dataset must be independently authorized.</p><p class="small muted">Reproduction requires the FlyRank research data-use permissions, which prohibit reidentification and redistributing raw records.</p></section>
<!-- EDITABLE AUTHOR SECTION: edit AUTHOR in the first Colab cell or this HTML after the notebook runs. -->
<section id="author"><div class="sectionnum">08 / Researcher &amp; identity</div><h2>About the author</h2><div class="ident"><div class="avatar" aria-hidden="true">$INITIALS</div><div><h3>$AUTHOR_NAME</h3><p><strong>$AUTHOR_POSITION</strong>$AFFILIATION</p><p>$AUTHOR_BIO</p><p>$AUTHOR_NOTE</p><div class="links">$AUTHOR_LINKS</div></div></div></section>
<section id="credit"><div class="sectionnum">09 / Acknowledgments and data credit</div><h2>Research provenance</h2><p><a href="https://flyrank.ai">Built on the FlyRank ML Internship dataset</a>. FlyRank AI provided the training context and pseudonymized research warehouse. The interpretation and methodological limitations are the author’s responsibility. Data-use and research-publication restrictions remain in effect.</p></section>
</main><footer><div class="shell">FlyRank ML Internship · Lane 2 / Research capstone · <a href="#abstract">Back to abstract ↑</a></div></footer>
</body></html>
''').safe_substitute(
      TITLE=title,SUBTITLE=subtitle,STATUS=escaped(status),AUTHOR_NAME=author_full,
      AUTHOR_POSITION=escaped(AUTHOR['position']),ABSTRACT=escaped(abstract),
      PROBLEM=escaped(PAPER_EDIT['problem_paragraph']),CONTRIBUTION=escaped(PAPER_EDIT['contribution_paragraph']),
      RELEASE=escaped(release),DATES=escaped(dates),ELIGIBILITY=escaped(eligibility),COUNTS=escaped(counts),
      FIGURES=figure_html,METRIC_ROWS=summary_rows,BOOT_ROWS=bootstrap_html,LATER=later_html,IMPORTANCE_FIG=importance_fig,
      REASONS=reasons_html,BAND_ROWS=bands_html,INITIALS=escaped(initials),
      BASE_RATE=f"{report['holdout_prevalence']:.2%}" if report else 'not evaluated',
      MAJORITY_RATE=f"{1-report['holdout_prevalence']:.2%}" if report else 'not evaluated',
      RULE_LIFT=f"{report['rule_baseline']['lift_at_50']:.2f}" if report else 'not evaluated',
      RF_LIFT=f"{report['random_forest']['lift_at_50']:.2f}" if report else 'not evaluated',
      RULE_TP=str(round(50*report['rule_baseline']['precision_at_50'])) if report else '',
      RULE_FP=str(50-round(50*report['rule_baseline']['precision_at_50'])) if report else '',
      RF_TP=str(round(50*report['random_forest']['precision_at_50'])) if report else '',
      RF_FP=str(50-round(50*report['random_forest']['precision_at_50'])) if report else '',
      AFFILIATION=affiliation,AUTHOR_BIO=escaped(bio),AUTHOR_NOTE=escaped(PAPER_EDIT['author_note']),
      AUTHOR_LINKS=links,REPO=escaped(repo),
    )
    target=Path(output_dir)/'docs/index.html'
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(page,encoding='utf-8')
    print('Paper:',target)
    return target

paper_path=build_paper(REPORT)

## 8. Export the public-safe publication package

The ZIP includes the eight-section Markdown report, measured HTML paper, four aggregate charts, and aggregate results JSON. It does not contain raw warehouse observations, client or content identifiers, queries, private ranked rows, or access credentials.


In [ ]:
# Export public-safe publication artifacts; the notebook source is supplied in the initial ZIP.
from zipfile import ZipFile, ZIP_DEFLATED
public_files=[
    'docs/index.html',
    'docs/assets/precision_at_k.png',
    'docs/assets/overall_ranking.png',
    'docs/assets/review_reasons.png',
    'docs/assets/feature_importance.png',
    'work/reports/capstone_results_public.json',
    'work/capstone_report.md',
]
for item in public_files:
    assert (OUT/item).exists(),f'Missing expected public artifact: {item}'
# Audit against accidental publication of Hugging Face access credentials.
for item in public_files:
    if item.endswith(('.html','.json')):
        source=(OUT/item).read_text(encoding='utf-8')
        if re.search(r'hf_[A-Za-z0-9]{16,}',source):
            raise ValueError(f'Possible HF token string in {item}: publication blocked.')

result_zip=OUT/'FlyRank_Final_Lane2_Publication_Outputs.zip'
with ZipFile(result_zip,'w',ZIP_DEFLATED) as archive:
    for item in public_files:
        archive.write(OUT/item,item)
assert ZipFile(result_zip).testzip() is None
print('Public-only results ZIP:',result_zip)
print('Next: update work/notebooks/capstone.ipynb in your repository, and copy docs/ and work/reports/ from the ZIP.')
try:
    from google.colab import files
    files.download(str(result_zip))
except ImportError:
    print('Download the ZIP above from your current Python environment.')

## 9. Register the live website address after deployment

Once the public GitHub Pages website is accessible, the following optional cell writes its verified direct URL to `submission/paper_url.txt`. The file must contain a single URL line; the repository URL is submitted separately.


In [ ]:
# Run this optional cell only AFTER GitHub Pages is live.
# Only enter a confirmed direct HTTPS paper URL after GitHub Pages deployment.
LIVE_PAPER_URL = ''
if LIVE_PAPER_URL.strip():
    parsed=urlparse(LIVE_PAPER_URL.strip())
    if parsed.scheme!='https' or not parsed.netloc or 'github.com/' in LIVE_PAPER_URL:
        raise ValueError('Provide the public, HTTPS webpage address, not your github.com repository URL.')
    (OUT/'submission/paper_url.txt').write_text(LIVE_PAPER_URL.strip()+'\n',encoding='utf-8')
    print('Created submission/paper_url.txt with exactly one URL line.')
else:
    print('Publishing is not yet complete. When Pages is live, set LIVE_PAPER_URL and rerun this cell.')

### Publication and reproducibility controls

This source notebook keeps the gated data inside the authorized Colab session, evaluates the fixed Random Forest and benchmark on identical observations, and exports the aggregate report and public paper together. Its notebook outputs are intentionally cleared before committing to the repository.

After running the analysis, copy the generated `docs/`, `work/capstone_report.md`, and `work/reports/capstone_results_public.json` into the repository. Deploy `docs/` through GitHub Pages and record the **verified** direct URL in `submission/paper_url.txt` before submitting the repository URL.
